## Cconstruir un dataset con lags y medias móviles
* partiendo de una tabla base con ventas diarias por artículo y sucursal. 
* Este es un paso crítico antes de alimentar cualquier modelo como LightGBM o XGBoost.

In [1]:
# Librerías necesarias
import base64
from io import BytesIO
import os
import shutil
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
import time
import getpass
import uuid
import warnings
import traceback
from datetime import datetime, timedelta

# Acceso a Datos
from dotenv import dotenv_values
import psycopg2 as pg2
import pyodbc
import sqlalchemy
from sqlalchemy import text

# Graficos y Paralelismo
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from io import BytesIO
import base64
from multiprocessing import Pool, cpu_count
import json
from statsmodels.tsa.holtwinters import ExponentialSmoothing, Holt
import ace_tools_open as tools

# Configuración global
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

# Cargar configuración DINAMIDA de acuerdo al entorno
from dotenv import dotenv_values
import os
import sys
ENV_PATH = os.environ.get("FORECAST_ENV_PATH", "C:/ETL/FORECAST/.env")  # Toma Producción si está definido, o la ruta por defecto
# Verificar si el archivo .env existe
if not os.path.exists(ENV_PATH):
    print(f"El archivo .env no existe en la ruta: {ENV_PATH}")
    print(f"Directorio actual: {os.getcwd()}")
    sys.exit(1)
    
secrets = dotenv_values(ENV_PATH)
folder = f"{secrets['BASE_DIR']}/{secrets['FOLDER_DATOS']}"

In [2]:
def Open_Diarco_Data(): 
    conn_str = f"dbname={secrets['PG_DB']} user={secrets['PG_USER']} password={secrets['PG_PASSWORD']} host={secrets['PG_HOST']} port={secrets['PG_PORT']}"
    #print (conn_str)
    for i in range(5):
        try:    
            conn = pg2.connect(conn_str)
            return conn
        except Exception as e:
            print(f'Error en la conexión: {e}')
            time.sleep(5)
    return None  # Retorna None si todos los intentos fallan


def Open_Conn_Postgres():
    conn_str = f"dbname={secrets['PGP_DB']} user={secrets['PGP_USER']} password={secrets['PGP_PASSWORD']} host={secrets['PGP_HOST']} port={secrets['PGP_PORT']}"
    for i in range(5):
        try:
            conn = pg2.connect(conn_str)
            return conn 
        except Exception as e:
            print(f"Error en la conexión, intento {i+1}/{5}: {e}")
            time.sleep(5)
    return None  # Retorna None si todos los intentos fallan

In [3]:
etiqueta = 'VENTA_BASAL'

folder = secrets["FOLDER_DATOS"]
archivo_datos = f'{folder}/{etiqueta}.csv'
archivo_articulos = f'{folder}/{etiqueta}_articulos.csv'
archivo_stock = f'{folder}/{etiqueta}_stock_sucursal.csv'

In [ ]:
# Aquí puedes incluir el código para generar los datos si no EXISTE el Archivo en el CACHE
conn = Open_Diarco_Data()

# --- VENTAS --- DIARCO + BARRIO ( En 2 Bases de Datos distintas )
# 3= ALMACEN, 5=GOLOSINAS, 6=BEBIDAS 7= LIMPIEZA, 8= PERFUMERIA, 4= ART.FIESTA, 2=FRESCOS, 10=NON-FOOD
query_ventas = f"""
   SELECT 
        v.f_venta AS fecha, 
        v.c_articulo AS sku_id, 
        v.c_sucu_empr AS local_id, 
        v.i_precio_venta AS precio_venta,
        v.q_unidades_vendidas AS venta_unidades,
        1.00 as factor_precio,
        v.c_familia AS categoria_id
    FROM src.t702_est_vtas_por_articulo v
    WHERE v.f_venta >= '20250101'
         AND v.c_familia IN ( 5, 6)

    UNION ALL
 
    SELECT 
        v.f_venta AS fecha, 
        v.c_articulo AS sku_id, 
        v.c_sucu_empr AS local_id, 
        v.i_precio_venta AS precio_venta,
        v.q_unidades_vendidas AS venta_unidades,
        1.00 as factor_precio,
        v.c_familia AS categoria_id
	FROM src.t702_est_vtas_por_articulo_dbarrio v
    WHERE v.f_venta >= '20250101'
         AND v.c_familia IN ( 5, 6)
    --- ORDER BY 1, 2, 3, 4
"""
df = pd.read_sql(query_ventas, conn) # type: ignore
if df.empty:
    print(f"⚠️ No se encontraron ventas DIARCO + BARRIO para  {etiqueta}.")

# Convertir columnas a minúsculas si hay datos
if not df.empty:

    # Asegurar que 'fecha' es de tipo datetime
    df['fecha'] = pd.to_datetime(df['fecha'])

    # Ordenar correctamente por agrupación y fecha
    df = df.sort_values(['sku_id', 'local_id', 'fecha'])

    # Lags de ventas anteriores
    lags = [1, 7, 14, 28]

    for lag in lags:
        df[f'venta_lag_{lag}'] = (
            df.groupby(['sku_id', 'local_id'])['venta_unidades']
            .shift(lag)
        )


    df['local_id'] = df['local_id'].astype(int)
    df['sku_id'] = df['sku_id'].astype(int)
    df['fecha'] = pd.to_datetime(df['fecha'])
    
    # Eliminar filas con NaN en 'fecha', 'codigo_articulo' o 'sucursal'
    #ventas = df.dropna(subset=['fecha', 'sku_id', 'sucurlocal_idsal'], how='all')
    #    #df.columns = df.columns.str.lower()
    # Transformar tipos de datos si hay datos
else:
    print(f"⚠️ No se encontraron ventas para {etiqueta} ni en DIARCO ni en BARRIO.")
    df = pd.DataFrame(columns=['fecha', 'sku_id', 'local_id', 'unidades'])  # DataFrame vacío con columnas esperadas




In [ ]:
# Medias móviles simples
ventanas = [7, 14, 28]

for window in ventanas:
    df[f'media_movil_{window}'] = (
        df.groupby(['sku_id', 'local_id'])['venta_basal']
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )

    df[f'std_movil_{window}'] = (
        df.groupby(['sku_id', 'local_id'])['venta_basal']
        .transform(lambda x: x.shift(1).rolling(window).std())
    )

In [ ]:
# Flag si fue fin de semana
df['es_fin_de_semana'] = df['dia_semana'].isin([5, 6]).astype(int)

# Flag feriados (requiere lista externa de feriados)
feriados = pd.to_datetime(['2025-01-01', '2025-05-25', '2025-12-25'])  # ejemplo
df['es_feriado'] = df['fecha'].isin(feriados).astype(int)

In [ ]:
df_modelo = df.dropna(subset=[col for col in df.columns if 'lag' in col or 'media_movil' in col])


## Resultado
El DataFrame df_modelo contiene todas las variables necesarias para entrenar un modelo como LightGBM. Las columnas importantes son:

* venta_basal → target
* venta_lag_1, venta_lag_7, venta_lag_28 → lags
* media_movil_7, media_movil_28, std_movil_7, etc.
* dia_semana, mes, semana_del_anio, es_feriado
* sku_id, local_id → identificadores categóricos

In [ ]:
# Guardar solo VENTAS 
df.to_csv(f'{folder}/{etiqueta}_Demanda.csv', index=False, encoding='utf-8')
print(f"---> Datos de Ventas guardados")